In [64]:
import os
from typing import List

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from ddgs import DDGS

from agents import Agent, Runner, function_tool, OpenAIChatCompletionsModel
from openai import AsyncOpenAI
import requests
from IPython.display import Markdown,display

In [ ]:
load_dotenv(override=True)

In [ ]:
API_KEY = os.getenv("MISTRAL_API_KEY")

if not API_KEY:
    raise ValueError("OPENAI_API_KEY is not set.")

In [ ]:
client = AsyncOpenAI(api_key=API_KEY,base_url="https://api.mistral.ai/v1")

In [ ]:
model_MI=OpenAIChatCompletionsModel(
    model="open-mixtral-8x7b",
    openai_client=client
)

In [ ]:
client = AsyncOpenAI(
    api_key=os.getenv("GEMINI_API_KEY_2"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
    )

In [ ]:
model_GM=OpenAIChatCompletionsModel(
    model="gemini-flash-latest",
    openai_client=client
)

In [ ]:
class ResearchPlan(BaseModel):
    topic: str = Field(
        description="The main research topic that needs to be investigated."
    )

    research_questions: List[str] = Field(
        description="The important questions that must be answered during the research."
    )

    search_queries: List[str] = Field(
        description="Specific search queries that should be used to find relevant information."
    )

    research_scope: str = Field(
        description="The scope and boundaries of the research."
    )

In [ ]:
class SearchResult(BaseModel):
    title: str = Field(
        description="The title of the search result."
    )

    url: str = Field(
        description="The URL of the source."
    )

    snippet: str = Field(
        description="A short description or relevant information from the source."
    )


In [ ]:
class ResearchNotes(BaseModel):
    topic: str = Field(
        description="The research topic."
    )

    key_findings: List[str] = Field(
        description="The important factual findings discovered during the research."
    )

    sources: List[SearchResult] = Field(
        description="The sources used to support the research findings."
    )

    unanswered_questions: List[str] = Field(
        description="Important questions that could not be answered during the research."
    )

In [ ]:
@function_tool
def search_web(query: str) -> List[SearchResult]:
    """
    Search the web using DuckDuckGo and return relevant search results.
    """

    try:
        with DDGS() as ddgs:
            results = list(
                ddgs.text(
                    query,
                    max_results=5,
                )
            )

        search_results = []

        for result in results:
            search_results.append(
                SearchResult(
                    title=result.get("title", ""),
                    url=result.get("href", ""),
                    snippet=result.get("body", ""),
                )
            )

        return search_results

    except Exception as error:
        raise RuntimeError(
            f"Web search failed: {error}"
        ) from error

In [ ]:
research_planner = Agent(
    name="Research Planner",
    model=model_MI,
    instructions="""
You are a professional research planning agent.

Your job is to analyze the user's research request and create
a clear research plan.

You must:

1. Identify the main research topic.
2. Create important research questions.
3. Generate multiple precise search queries.
4. Define the scope of the research.
5. Avoid answering the research question yourself.
6. Focus only on planning the research.

Return your response using the ResearchPlan structured output.
""",
    output_type=ResearchPlan,
)


In [ ]:
search_agent = Agent(
    name="Research Search Agent",
    model=model_GM,
    instructions="""
You are a professional research agent.

Your job is to execute the research plan provided by the
Research Planner.

You have access to a web search tool.

Follow these rules:

1. Analyze the research plan.
2. Search the web using the provided search queries.
3. Use the search tool when additional information is required.
4. Collect relevant and trustworthy information.
5. Remove duplicate information.
6. Do not invent facts.
7. Preserve the source title, URL, and useful snippet.
8. Identify important findings.
9. Identify questions that remain unanswered.
10. Return structured research notes.

Do not write the final report.

Return your response using the ResearchNotes structured output.
""",
    tools=[search_web],
    output_type=ResearchNotes,
)


In [ ]:
writer_agent = Agent(
    name="Research Writer",
    model=model_MI,
    instructions="""
You are a professional research writer.

Your job is to transform structured research notes into
a clear, accurate and professional research report.

Follow these rules:

1. Use only the information provided in the research notes.
2. Do not invent facts.
3. Organize information logically.
4. Use Markdown headings.
5. Use bullet points when appropriate.
6. Explain important findings clearly.
7. Include a conclusion.
8. Include a Sources section.
9. Preserve the provided source URLs.
10. Clearly mention unanswered questions when relevant.

The final response should be a polished research report.
""",
)

In [ ]:
BREVO_API_KEY = os.getenv("BREVO_API_KEY")
SENDER_EMAIL = os.getenv("SENDER_EMAIL")
SENDER_NAME = os.getenv("SENDER_NAME")
if not BREVO_API_KEY:
    raise ValueError("BREVO_API_KEY is not set.")

if not SENDER_EMAIL:
    raise ValueError("SENDER_EMAIL is not set.")

In [ ]:
@function_tool
def send_email(
    recipient_email: str,
    subject: str,
    content: str,
) -> str:
    """
    Send an email using the Brevo transactional email API.
    """

    url = "https://api.brevo.com/v3/smtp/email"

    headers = {
        "accept": "application/json",
        "api-key": BREVO_API_KEY,
        "content-type": "application/json",
    }

    payload = {
        "sender": {
            "name": SENDER_NAME or "Research Agent",
            "email": SENDER_EMAIL,
        },
        "to": [
            {
                "email": recipient_email,
            }
        ],
        "subject": subject,
        "htmlContent": content,
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=30,
    )

    if response.status_code not in (200, 201, 202):
        raise RuntimeError(
            f"Brevo email failed: {response.status_code} - "
            f"{response.text}"
        )

    return "Email sent successfully."



In [ ]:
email_agent = Agent(
    name="Research Email Agent",
    instructions="""
You are a professional research email agent.

Your job is to take a completed research report and send it
to the specified recipient using the send_email tool.

Follow these rules:

1. Read the complete research report.
2. Do not change or invent research facts.
3. Create a concise and professional email subject.
4. Convert the Markdown research report into clean HTML.
5. Preserve the report's headings, paragraphs, bullet points,
   and important information.
6. Include the complete research report in the email.
7. Use the provided recipient email address.
8. Send the email using the send_email tool.
9. Do not claim that the email was sent unless the tool
   successfully confirms it.
10. After successful delivery, return a short confirmation.

You must use the send_email tool to actually send the email.
""",
    tools=[send_email],
    model=model_MI
)

In [ ]:
user_query="what is the future of autonomius ai agents"

In [ ]:
planner_result = await Runner.run(
    research_planner,
    user_query,
    )
research_plan = planner_result.final_output
print("\n=== RESEARCH PLAN ===")
print(research_plan.model_dump_json(indent=2))

In [ ]:
search_result = await Runner.run(
    search_agent,
    research_plan.model_dump_json(),
    )
research_notes = search_result.final_output

In [ ]:
writer_result = await Runner.run(
    writer_agent,
    research_notes.model_dump_json(),
    )
report=writer_result.final_output

In [ ]:
prompt = f"""
Send the following research report to this recipient:

Recipient:
"hariskhann502@gmail.com"

Research Report:
{report}
"""

In [ ]:
result = await Runner.run(
    email_agent,
    prompt,
)


In [65]:
display(Markdown(result.final_output))

The email containing the research report has been successfully sent to "hariskhann502@gmail.com".